# LoRA Fine Tuning with QLoRA for Decoder model (Qwen-2.5-3B-Instruct) 

We will implement the complete LoRA (Low Rank Adaptation) fine tuning pipeline used in the intent classifier project. The script takes a pre trained causal language model (like Qwen 2.5 3B Instruct) and fine tunes it with QLoRA to classify user intents from conversational data.

## What you will learn
- How QLoRA quantization reduces GPU memory usage
- How to load and configure a base model for fine tuning
- How to transform enterprise data schemas into chat templates for SFTTrainer
- How to configure LoRA adapters (rank, alpha, target modules)
- How SFTConfig controls the training loop, evaluation and checkpointing
- How MLflow lineage callbacks track data provenance
- How multi GPU (DDP) training changes the setup
- How to save the final adapter for downstream evaluation

In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0 bitsandbytes==0.48.1
import os

!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 216.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.4
    Uninstalling datasets-4.8.4:
      Successfully uninstalled datasets-4.8.4

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/red-hat-data-services/mlflow (to revision rhoai-3.3) to /tmp/pip-req-build-o3lmadlu
  Running command git clone --filter=blo

In [2]:
import os

os.environ["MLFLOW_TRACKING_TOKEN"] = "sha256~Homqs4edkiZJLG2KiXVatFjoXHJ8lwDVBgWcj4F-FPQ"
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "decoder-sft"
os.environ["MLFLOW_RUN_NAME"] = "decoder-sft-run"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"decoder-sft\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "decoder-finetuning"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


---
## 1 Imports and Dependencies

The script pulls together several libraries from the Hugging Face ecosystem along with MLflow for experiment tracking. Here is what each one provides:

| Library | Purpose |
|---|---|
| `transformers` | Base model loading (`AutoModelForCausalLM`), tokenizer, quantization config |
| `peft` | LoRA configuration (`LoraConfig`) for parameter efficient fine tuning |
| `trl` | `SFTTrainer` and `SFTConfig` for supervised fine tuning on chat formatted data |
| `datasets` | Hugging Face dataset loading and splitting utilities |
| `bitsandbytes` | Backend for 4 bit quantization (used through `BitsAndBytesConfig`) |
| `mlflow` | Experiment tracking, parameter logging and dataset lineage |
| `torch` | PyTorch tensors and dtype configuration |

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import shutil
import os
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from datetime import datetime
import json
import mlflow
from transformers import TrainerCallback

/opt/app-root/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 2 MLflow Lineage Callback

When HF Trainer runs with `report_to="mlflow"`, it automatically creates an MLflow run and logs the standard hyperparameters (learning rate, batch size, epochs, etc.). However, it does not know about our custom metadata like the S3 data source or the LoRA specific hyperparameters.

This custom callback hooks into the `on_train_begin` event to inject that extra context into the same MLflow run that Hugging Face created. This gives us full data lineage, so we can trace every trained adapter back to the exact dataset version and LoRA config that produced it.

The `state.is_world_process_zero` guard is critical for multi GPU training: without it, every GPU rank would try to log the same params, causing duplicate parameter errors in MLflow.

In [4]:
class MLflowLineageCallback(TrainerCallback):
    """
    custom callback to inject S3 dataset lineage and LoRA hyperparameters 
    into the MLflow run automatically created by Hugging Face Trainer
    """
    def __init__(self, custom_args, peft_config, train_ds, val_ds, test_ds):
        self.custom_args = custom_args
        self.peft_config = peft_config
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds

    def on_train_begin(self, args, state, control, **kwargs):
        # only execute on the main process (Rank 0) 
        if state.is_world_process_zero:
            print("Injecting S3 data lineage into Hugging Face's active MLflow run...")
            
            # HF automatically logs learning_rate, batch_size, epochs, etc.
            # so we omit them here to prevent MLflow "duplicate param" errors
            mlflow.log_params({
                "model_name": self.custom_args.model_name,
                "data_path": self.custom_args.data_path,
                "lora_r": self.peft_config.r,
                "lora_alpha": self.peft_config.lora_alpha,
                "lora_dropout": self.peft_config.lora_dropout,
            })
            
            # convert datasets and log them directly into HF's run
            ds_train = mlflow.data.from_pandas(
                self.train_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_train"
            )
            ds_val = mlflow.data.from_pandas(
                self.val_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_val"
            )
            ds_test = mlflow.data.from_pandas(
                self.test_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_test"
            )
            
            mlflow.log_input(ds_train, context="training")
            mlflow.log_input(ds_val, context="evaluation")
            mlflow.log_input(ds_test, context="test")

---
## 3 Data Formatting: Enterprise Schema to Chat Template

The training data arrives in an enterprise schema with fields like `message`, `intent` and `session_history`. But SFTTrainer expects data in the standard chat `messages` format (a list of role/content dicts).

This function bridges that gap by:

1. Adding a system prompt that tells the model it is an intent classifier (skipped for Gemma which does not support system prompts)
2. Converting session history turns (`bot` -> `assistant`, `user` -> `user`)
3. Appending the current user message
4. Appending the target assistant response (`{"intent": "ABCD"}`) so the model learns to generate it during training

### Input format (enterprise schema)
```json
{
    "message": "now do this",
    "intent": "ABCD",
    "session_history": [
        {"role": "bot", "message": "something"},
        {"role": "user", "message": "something else"}
    ]
}
```

### Output format (SFTTrainer chat template)
```json
{
    "messages": [
        {"role": "system", "content": "You are an AI intent classifier..."},
        {"role": "assistant", "content": "something"},
        {"role": "user", "content": "something else"},
        {"role": "user", "content": "now do this"},
        {"role": "assistant", "content": "{\"intent\": \"ABCD\"}"}
    ]
}
```

In [5]:
def format_messages_for_training(row: dict, base_model_id: str) -> dict:
    """
    translates the enterprise data schema into a model agnostic chat template for SFTTrainer
    """
    messages = []
    system_prompt = (
        "You are an AI intent classifier. Analyze the conversation history "
        "and the latest user message. You must output only a valid JSON object "
        "containing the predicted 'intent'."
    )
    
    model_name = base_model_id.lower()
    
    # handle model-specific system prompt support
    if "qwen" in model_name or "phi" in model_name or "deepseek" in model_name:
        messages.append({"role": "system", "content": system_prompt})
    elif "gemma" in model_name:
        pass 
    else:
        messages.append({"role": "system", "content": system_prompt})

    # process session history
    for turn in row.get("session_history", []):
        print (turn)
        hf_role = "assistant" if turn.get("role") == "assistant" else "user"
        messages.append({"role": hf_role, "content": turn.get("content", "")})
        
    # process current message
    current_msg = row.get("user_message", "")
    if "gemma" in model_name and len(messages) == 0:
        current_msg = f"{system_prompt}\n\n{current_msg}"
        
    messages.append({"role": "user", "content": current_msg})
    
    # critical for training: append the target output so the model learns to generate it
    target_output = json.dumps({"intent": row.get("intent", "UNKNOWN")})
    messages.append({
        "role": "assistant",
        "content": target_output
    })
    
    return {"messages": messages}

Let's test the formatting function with a sample row to see the transformation in action.

In [6]:
# test the formatter with a sample row
sample_row = {
    "user_message": "I want to change my shipping address",
    "intent": "UPDATE_SHIPPING",
    "session_history": [
        {"role": "assistant", "content": "Hello! How can I help you today?"},
        {"role": "user", "content": "I placed an order yesterday"}
    ]
}

# test with Qwen (supports system prompt)
result = format_messages_for_training(sample_row, "Qwen/Qwen2.5-3B-Instruct")
print("=== Qwen formatted output ===")
for msg in result["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

print()

# test with Gemma (no system prompt, prepended to first user message)
sample_no_history = {"message": "check my balance", "intent": "CHECK_BALANCE", "session_history": []}
result_gemma = format_messages_for_training(sample_no_history, "google/gemma-2b")
print("=== Gemma formatted output (no history) ===")
for msg in result_gemma["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

{'role': 'assistant', 'content': 'Hello! How can I help you today?'}
{'role': 'user', 'content': 'I placed an order yesterday'}
=== Qwen formatted output ===
  [system]: You are an AI intent classifier. Analyze the conversation history and the latest
  [assistant]: Hello! How can I help you today?
  [user]: I placed an order yesterday
  [user]: I want to change my shipping address
  [assistant]: {"intent": "UPDATE_SHIPPING"}

=== Gemma formatted output (no history) ===
  [user]: You are an AI intent classifier. Analyze the conversation history and the latest
  [assistant]: {"intent": "CHECK_BALANCE"}


---
## 4 Configuration and Setup

In the production script, these values come from `argparse` command line arguments. For the notebook, we define them inline. The key parameters are:

| Parameter | What it controls |
|---|---|
| `model_name` | Which pre trained model to fine tune |
| `data_path` | Path to the JSON training data |
| `output_dir` | Where to write the saved LoRA adapter |
| `dataset_source` | S3 URI logged to MLflow for data lineage |
| `run_name` | Name of the MLflow experiment run |

The `LOCAL_RANK` environment variable is set automatically by PyTorch's distributed launcher (`torchrun`). Rank 0 is the "main" process that handles housekeeping tasks like cleaning output directories and saving the final adapter.

In [7]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
data_path = "./mnt/data/training_dataset.json"      # path to JSON training data
output_dir = "./mnt/data/adapters"
dataset_source = ""                               # s3 uri for MLflow lineage
run_name = "lora-finetune-run"

# multi GPU rank: set by torchrun, defaults to 0 for single GPU
local_rank = int(os.environ.get("LOCAL_RANK", 0))
print(f"local_rank: {local_rank}")

# rank 0 housekeeping: clean old output to avoid adapter corruption
if local_rank == 0:
    if os.path.exists(output_dir):
        print(f"cleaning existing output dir: {output_dir}")
        shutil.rmtree(output_dir)

local_rank: 0


---
## 5 QLoRA: 4 Bit Quantization Config

QLoRA (Quantized LoRA) is the technique that makes it possible to fine tune a 7B parameter model on a single 16GB GPU (like the T4 in ARO NC64as_T4_v3 instances). It works by loading the base model weights in 4 bit precision instead of the standard 16 bit, reducing VRAM from ~16GB down to ~6GB.

### BitsAndBytesConfig breakdown

| Setting | Value | What it does |
|---|---|---|
| `load_in_4bit` | `True` | Activates 4 bit quantization for all model weights |
| `bnb_4bit_quant_type` | `"nf4"` | Uses NormalFloat4, a data type optimized for normally distributed neural network weights |
| `bnb_4bit_use_double_quant` | `True` | Quantizes the quantization constants themselves, saving additional memory (important for multi GPU) |
| `bnb_4bit_compute_dtype` | `torch.float16` | Computation happens in float16 for speed, even though weights are stored in 4 bit |

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"quantization config: {bnb_config}")

quantization config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



---
## 6 Loading the Base Model

The base model is loaded with the quantization config applied. A few things to note about the device mapping:

- For single GPU training, `device_map="auto"` works fine and lets HF decide placement
- For multi GPU DDP training, we pin each process to its own GPU using `device_map={"":local_rank}`. This avoids the "all layers on GPU 0" problem where `auto` would try to shard the model across GPUs instead of giving each DDP rank its own full copy

Setting `use_cache=False` disables KV cache, which is incompatible with gradient checkpointing (both try to manage the same intermediate activations).

In [9]:
print(f"loading model: {model_name}...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, 
    device_map={"":local_rank},
    torch_dtype=torch.float16, 
    trust_remote_code=True 
)

# CRITICAL: Force the base config to fp16 so PEFT doesn't secretly initialize LoRA weights in bfloat16
model.config.torch_dtype = torch.float16
model.config.use_cache = False


`torch_dtype` is deprecated! Use `dtype` instead!


loading model: Qwen/Qwen2.5-3B-Instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]/opt/app-root/lib64/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 2/2 [00:46<00:00, 23.04s/it]


---
## 7 Tokenizer Setup and Chat Template Override

The tokenizer converts text into token IDs that the model understands. Two important adjustments are made here:

### Padding token fix
Many models (including Qwen) do not ship with a dedicated padding token. We set `pad_token = eos_token` so that the tokenizer can pad shorter sequences in a batch without crashing. This is safe because the attention mask ensures the model ignores padding positions.

### Chat template override
The default Qwen tokenizer ships with a chat template that does not include TRL's `{% generation %}` markers. SFTTrainer relies on these markers to know which tokens belong to the assistant's response (the part we want to compute loss on when `assistant_only_loss=True`). Without this override, the trainer would either compute loss on the entire sequence or fail outright.

The template uses ChatML format:
```
<|im_start|>system
You are an AI intent classifier...<|im_end|>
<|im_start|>user
check my balance<|im_end|>
<|im_start|>assistant
{"intent": "CHECK_BALANCE"}<|im_end|>
```

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# override with a TRL-compatible ChatML template that includes generation markers
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{% generation %}{{ message['content'] }}{% endgeneration %}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

print(f"pad_token: {tokenizer.pad_token}")
print(f"vocab size: {tokenizer.vocab_size}")

pad_token: <|im_end|>
vocab size: 151643


---
## 8 Dataset Loading and 80/10/10 Split

The training data is loaded from a JSON file and split into three sets:

| Split | Percentage | Purpose |
|---|---|---|
| Train | 80% | Used to update the LoRA weights |
| Validation | 10% | Used during training to detect overfitting (eval_loss) |
| Test | 10% | Quarantined and saved to disk. Never seen during training, used only for final evaluation |

The splitting is done in two steps:
1. First split: 80% train, 20% remainder
2. Second split: The 20% remainder is split 50/50 into validation (10% of total) and test (10% of total)

After splitting, the `format_messages_for_training` function is applied via `.map()` to convert every row from the enterprise schema into the chat template format. The original columns (`message`, `intent`, `session_history`) are dropped with `remove_columns` to free memory, since they have been folded into the `messages` column.

In [13]:
data_path = "../mnt/data/training_dataset.json"
dataset = load_dataset("json", data_files=data_path, split="train")

# 80/10/10 split
split_1 = dataset.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")

# separate the test set so it is never touched during training
if local_rank == 0:
    print("saving quarantined test set to ../mnt/data/test_split.jsonl...")
    test_dataset.to_json("../mnt/data/test_split.jsonl")

Generating train split: 466 examples [00:00, 11152.07 examples/s]


train: 372 | val: 47 | test: 47
saving quarantined test set to ../mnt/data/test_split.jsonl...


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 207.14ba/s]


In [14]:
# apply the chat template transformation to train and val sets
print("formatting dataset for the specific model...")

train_dataset = train_dataset.map(
    lambda row: format_messages_for_training(row, model_name), 
    remove_columns=["user_message", "intent", "session_history"]
)

val_dataset = val_dataset.map(
    lambda row: format_messages_for_training(row, model_name), 
    remove_columns=["user_message", "intent", "session_history"]
)
# remove_columns drops the original fields after they have been folded
# into the 'messages' column, freeing memory

print(f"train columns after formatting: {train_dataset.column_names}")
print(f"sample formatted row:")
print(json.dumps(train_dataset[0], indent=2))

formatting dataset for the specific model...


Map: 100%|██████████| 372/372 [00:00<00:00, 3045.45 examples/s]


{'content': 'ซื้อซิมใหม่มาแต่ยังใช้งานไม่ได้ครับ ลองทำตามขั้นตอนในเน๊จก้อยังไม่ได้ เหนื่อยใจ', 'role': 'user'}
{'content': 'เป็นซิมเติมเงินหรือรายเดือนคะ', 'role': 'assistant'}
{'content': 'เติมเงินครับ เออลองดูแพ๊กเน็ดแรงๆ ให้ด้วย', 'role': 'user'}
{'content': 'ถ้าเป็นซิมเติมเงินใหม่ สามารถช่วยเปิดใช้งานให้ได้ ต้องการให้ดำเนินการเลยไหมคะ [option: {ทันที, รอการเปิดใช้งาน}]', 'role': 'assistant'}
{'content': 'เคยสมัคริงโทนตั้งแต่เมื่อไรอ่ะ 🎵 เบอร์ 0xx-xxx-xxxx', 'role': 'user'}
{'content': 'ตรวจสอบแล้วค่ะ หมายเลข 0xx-xxx-xxxx สมัครริงโทนครั้งแรกเมื่อวันที่ 1 ตุลาคม 2568 ค่ะ เพลง "ลมหนาว" 29 บาท/เดือน ใช้มา 6 เดือนแล้วค่ะ', 'role': 'assistant'}
{'content': 'นานเหมือนกันนะ 😲 เคยเปลี่ยนเพลงบ้างมั้ย', 'role': 'user'}
{'content': 'เปลี่ยน 1 ครั้งค่ะ เมื่อวันที่ 10 มี.ค. 2569 จาก "ลมหนาว" เป็น "คิดถึง" ค่ะ', 'role': 'assistant'}
{'content': 'เข้าใจค่ะ มีแพ็กโรมมิ่งราคาประหยัดให้เลือกด้วยนะคะ [options :{ 1GB 99 บาท, 2GB 149 บาท}] ค่ะ', 'role': 'assistant'}
{'content': 'รู้สึกว่า net โรมมิ่งไม่

Map: 100%|██████████| 47/47 [00:00<00:00, 2942.14 examples/s]

{'content': 'ได้ค่ะ รบกวนขอทราบหมายเลขโทรศัพท์ก่อนค่ะ', 'role': 'assistant'}
{'content': '06xxxxxxxx', 'role': 'user'}
{'content': 'มีบิลประจำเดือนนี้ในระบบแล้วค่ะ', 'role': 'assistant'}
{'content': 'ครับ', 'role': 'user'}
{'content': 'ต้องการข้อมูลส่วนใดเพิ่มเติมหรือไม่คะ', 'role': 'assistant'}
{'content': 'ตรวจสอบค่าใช้จ่ายแพ็กเกจเน็ตของหมายเลข 0xx-xxx-xxxx ย้อนหลัง 3 เดือนค่ะ:\n- มี.ค. 2569: แพ็กหลัก 349 บาท + เน็ตเสริม 79 บาท = 428 บาท\n- ก.พ. 2569: แพ็กหลัก 349 บาท = 349 บาท\n- ม.ค. 2569: แพ็กหลัก 349 บาท + เน็ตเสริม 99 บาท = 448 บาท\nรวม 3 เดือน = 1,225 บาทค่ะ', 'role': 'assistant'}
{'content': 'เยอะเหมือนกันนะคะ เฉลี่ยเดือนละเท่าไหร่', 'role': 'user'}
{'content': 'เฉลี่ยเดือนละ 408 บาทค่ะ ถ้าไม่อยากซื้อเน็ตเสริมบ่อยๆ แนะนำอัปเกรดเป็นแพ็กเกจ 20GB ราคา 399 บาท/เดือน จะคุ้มกว่าค่ะ', 'role': 'assistant'}
{'content': 'น่าสนใจเหมือนกัน แต่ขอดูก่อนค่ะ ขอบคุณนะคะ', 'role': 'user'}
{'content': 'ได้เลยค่ะ หากต้องการเปลี่ยนแพ็กเกจหรือสอบถามเพิ่มเติม ติดต่อได้ตลอดนะคะ ขอบคุณค่ะ 😊', 'role': 

---
## 9 LoRA Configuration

LoRA (Low Rank Adaptation) freezes the original model weights and injects small trainable matrices into specific layers. Instead of updating all 7 billion parameters, we only train a few million, making fine tuning much faster and cheaper.

### Key parameters

| Parameter | Value | Explanation |
|---|---|---|
| `r` | 16 | Rank of the low rank matrices. Higher rank = more capacity but slower and more memory. 16 is a good balance for classification tasks |
| `lora_alpha` | 32 (2x r) | Scaling factor for the LoRA update. The effective update is scaled by `alpha/r`. Setting alpha to 2x rank means the effective multiplier is 2.0 |
| `lora_dropout` | 0.05 | Regularization to prevent overfitting the small adapter |
| `bias` | `"none"` | Do not train bias terms (standard for LoRA) |
| `task_type` | `"CAUSAL_LM"` | Tells PEFT this is a causal language model (autoregressive, left to right) |
| `target_modules` | attention + MLP projections | Which linear layers get the LoRA adapters injected |

### Target modules explained
The 7 target modules cover both the attention mechanism and the feed forward (MLP) block:
- `q_proj`, `k_proj`, `v_proj`, `o_proj`: the four projections in multi head attention (query, key, value and output)
- `gate_proj`, `up_proj`, `down_proj`: the three projections in the SwiGLU feed forward block

In [15]:
r = 16
peft_config = LoraConfig(
    r=r,
    lora_alpha=(2 * r),
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

---
## 10 Training Arguments (SFTConfig)

SFTConfig extends Hugging Face's `TrainingArguments` with extra fields for supervised fine tuning. This is where the bulk of the training behavior is controlled.

### Sequence and batch settings
| Setting | Value | Rationale |
|---|---|---|
| `max_length` | 1024 | Maximum token length per example. Reduced from 2048 for multi GPU memory savings |
| `per_device_train_batch_size` | 1 | Reduced to 1 for multi GPU to fit within per GPU VRAM |
| `gradient_accumulation_steps` | 4 | With 4 GPUs x 1 batch x 4 accumulation steps = 16 effective global batch size |
| `packing` | False | Do not pack multiple examples into one sequence (keeps example boundaries clean) |

### Learning rate schedule
| Setting | Value | Rationale |
|---|---|---|
| `learning_rate` | 2e-4 | Standard starting LR for LoRA fine tuning |
| `warmup_ratio` | 0.1 | Spends the first 10% of training linearly ramping up the LR to prevent early instability |
| `lr_scheduler_type` | "cosine" | After warmup, LR follows a cosine decay curve to zero |
| `max_grad_norm` | 0.3 | Clips gradients to prevent exploding updates |

### Evaluation and checkpointing
| Setting | Value | Rationale |
|---|---|---|
| `eval_strategy` | "steps" | Run validation at regular step intervals |
| `eval_steps` | 0.1 | Evaluate every 10% of total training steps |
| `save_strategy` / `save_steps` | "steps" / 0.1 | Save checkpoints at the same interval |
| `save_total_limit` | 2 | Keep only the 2 most recent checkpoints to save disk |
| `load_best_model_at_end` | True | After training, roll back to the checkpoint with the lowest eval_loss (anti overfitting) |

### SFT specific settings
| Setting | Value | Rationale |
|---|---|---|
| `assistant_only_loss` | True | Only compute loss on the assistant's response tokens, not the conversation history. This is critical for intent classification because we want the model to learn the intent output, not memorize the prompt |

### Multi GPU (DDP) settings
| Setting | Value | Rationale |
|---|---|---|
| `gradient_checkpointing` | True | Trades compute for memory by recomputing activations during backward pass instead of storing them |
| `gradient_checkpointing_kwargs` | `{'use_reentrant': False}` | Avoids the PyTorch reentrant autograd bug with DDP |
| `ddp_find_unused_parameters` | False | Speeds up DDP by skipping the scan for unused parameters (safe because LoRA uses all parameters) |
| `per_device_eval_batch_size` | 1 | Keeps evaluation memory low on each GPU |
| `eval_accumulation_steps` | 1 | Moves evaluation results off GPU to CPU after every step to prevent OOM |

In [16]:
training_args = SFTConfig(
    output_dir=output_dir,
    max_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # Increased for stability
    learning_rate=1e-4, # Lowered to prevent mode collapse
    logging_steps=10, # Log more frequently for visibility
    num_train_epochs=8, # Increased for JSON muscle memory
    max_grad_norm=0.3,
    warmup_ratio=0.15, # Gentler ramp-up
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False, # Explicitly deny bfloat16 to prevent T4 crashes
    push_to_hub=False,
    packing=False,
    assistant_only_loss=True, # <-- Critical for intent classification

    # mlflow tracking
    report_to="mlflow",
    run_name=run_name,

    # evaluation and checkpointing
    eval_strategy="epoch", # Evaluate at the end of each epoch
    eval_steps=0.1,
    save_strategy="epoch", # Evaluate at the end of each epoch
    save_steps=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Memory optimizations
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}, 
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
)

---
## 11 Initialize Trainer and Start Training

The `SFTTrainer` ties everything together:
- The quantized base model
- The LoRA adapter config (injected automatically via `peft_config`)
- The formatted train/eval datasets
- The tokenizer (passed as `processing_class`)
- The MLflow lineage callback

When `.train()` is called, the trainer will:
1. Apply the chat template to tokenize each example
2. Compute loss only on assistant tokens (due to `assistant_only_loss=True`)
3. Update only the LoRA matrices (the base model weights stay frozen)
4. Log metrics to MLflow every 25 steps
5. Run validation every 10% of steps and checkpoint the model
6. At the end, roll back to the best checkpoint based on eval_loss

In [17]:
from types import SimpleNamespace
args_ns = SimpleNamespace(
    model_name=model_name,
    data_path=data_path,
    dataset_source=dataset_source
)

lineage_callback = MLflowLineageCallback(
    custom_args=args_ns,
    peft_config=peft_config,
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config, 
    processing_class=tokenizer,
    callbacks=[lineage_callback]
)

print("starting training...")
trainer.train()

Truncating eval dataset: 100%|██████████| 47/47 [00:00<00:00, 11598.75 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


starting training...


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

Injecting S3 data lineage into Hugging Face's active MLflow run...


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

Epoch,Training Loss,Validation Loss
1,1.930600,1.379561
2,1.015900,0.758311
3,0.616100,0.608718
4,0.467000,0.562913
5,0.354000,0.559620
6,0.261300,0.567117
7,0.209100,0.581680
8,0.182700,0.587085


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

🏃 View run lora-finetune-run at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/decoder-sft/experiments/6/runs/fa71eb22b8f6440a98450da86ea33aa6
🧪 View experiment at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/decoder-sft/experiments/6


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


TrainOutput(global_step=192, training_loss=0.6792114164369801, metrics={'train_runtime': 2588.6379, 'train_samples_per_second': 1.15, 'train_steps_per_second': 0.074, 'total_flos': 1.1558580338884608e+16, 'train_loss': 0.6792114164369801})

---
## 12 Saving the LoRA Adapter

After training completes, we save only the LoRA adapter weights (not the full base model). The saved adapter is a small directory containing:
- `adapter_model.safetensors`: the trained LoRA matrices
- `adapter_config.json`: the LoRA hyperparameters needed to reload

The adapter is saved to `output_dir/latest` so that downstream evaluation scripts can always find the most recent adapter at a predictable path.

Only rank 0 saves the adapter in multi GPU training. The HF Trainer handles DDP synchronization internally, so all ranks have converged to the same weights before rank 0 writes to disk.

In [18]:
# save the adapter to a predictable path for the evaluator
adapter_output_dir = f"{output_dir}/latest"

print(f"rank {local_rank}: training complete, saving adapter...")
if local_rank == 0:
    trainer.save_model(adapter_output_dir)
    print(f"adapter saved to: {adapter_output_dir}")
    
    # verify the saved files
    if os.path.exists(adapter_output_dir):
        saved_files = os.listdir(adapter_output_dir)
        print(f"saved files: {saved_files}")

rank 0: training complete, saving adapter...
adapter saved to: ./mnt/data/adapters/latest
saved files: ['added_tokens.json', 'adapter_config.json', 'chat_template.jinja', 'README.md', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'merges.txt', 'vocab.json']


---
## Summary

We have done the complete LoRA fine tuning pipeline:

1. QLoRA quantization to fit a 3B model in 16GB of VRAM
2. Enterprise data schema transformation into HF chat templates
3. 80/10/10 dataset splitting with a quarantined test set
4. LoRA adapter injection into attention and MLP layers
5. SFTTrainer with assistant only loss for intent classification
6. MLflow lineage tracking for full data provenance
7. Multi GPU DDP considerations (device mapping, gradient checkpointing, rank 0 guards)
8. Adapter saving for downstream evaluation

The production version of this implementation for a bigger model that will not fit into one single GPU should be run via `torchrun` for multi GPU training:
```bash
torchrun --nproc_per_node=4 lora-adapter-builder.py \
    --model_name Qwen/Qwen2.5-3B-Instruct \
    --data_path /mnt/data/training_data.json \
    --output_dir /mnt/data/adapters \
    --dataset_source s3://bucket/data/v1 \
    --run_name lora-intent-v1
```